In [78]:
import pandas as pd

#reading dynamic data from the google form used for data collection.
excel_url = "https://docs.google.com/spreadsheets/d/1r6E68lyiUEvV0gx1HkQS1EkL_OjyOrtwcKsYi4jsuqk/export?format=csv" 

df = pd.read_csv(excel_url)

#step 1: fix colomn namings.(lowercase, nospace, no ambiguty)
df.columns = (
    df.columns
    .str.strip()                 # remove leading/trailing spaces
    .str.lower()                 # lowercase everything
    .str.replace(' ', '_')       # replace spaces with underscore
)

# rename a miss spelled name:
df = df.rename(columns={
    'stoped_using': 'stopped_using'
})

# clean the inner cell inputs
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# solving the scientify value messing the phone number.
df['phone'] = (df['phone'].astype(str)).str.replace('.0', '', regex=False)



#Step 2: Dealing with the missing values.


# scam story high invalid values
df['scam_story'] = df['scam_story'].fillna('no_story')
df['trust_insight'] = df['trust_insight'].fillna('no_input')



# report about the missing values.
missing_report = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().sum() / len(df)) * 100
})
missing_report
# option to safe into reports,
# missing_report.to_csv('../reports/missing_report.csv', index=True)

# catagorise the scamming responses.
df['scam_experience_clean'] = df['scam_experience'].replace({
    'No, but I worry about it': 'worried_but_no_scam',
    'Almost — I caught it in time': 'almost_scammed',
    'No, never': 'never_scammed',
    'Yes': 'scammed'
})

# df['scam_experience_clean'].value_counts(dropna=False)

# catagorise the safety feeling.
df['safety_feeling_clean'] = (
    df['safety_feeling']
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# create a number scaling for easy analysis.
safety_score_map = {
    'very_unsafe': 1,
    'unsafe': 2,
    'neutral': 3,
    'safe': 4,
    'very_safe': 5
}

df['safety_score'] = df['safety_feeling_clean'].map(safety_score_map)


# CLEAN TOP CONCERN COLOMN.
df['top_concerns_clean'] = (
    df['top_concerns']
    .str.lower()
    .str.replace('/', '')
    .str.strip()
)

# PRESENT EACH OPTION WITH A NUMBER FOR EASY ANALYSIS.
df['concern_fake_sellers'] = df['top_concerns_clean'].str.contains('fake sellers', na=False)
df['concern_payment_fraud'] = df['top_concerns_clean'].str.contains('payment fraud', na=False)
df['concern_meeting_strangers'] = df['top_concerns_clean'].str.contains('meeting strangers', na=False)
df['concern_product_not_described'] = df['top_concerns_clean'].str.contains('product not as described', na=False)

# CONVERT THIS INTO 0/1 FOR IT TO GIVE STATS ON CONCERNS
df[['concern_fake_sellers',
    'concern_payment_fraud',
    'concern_meeting_strangers',
    'concern_product_not_described']] = \
df[['concern_fake_sellers',
    'concern_payment_fraud',
    'concern_meeting_strangers',
    'concern_product_not_described']].astype(int)
# GET THE AVARAGE OF PEOPLE CONCERNED BY EACH FACTOR.
# df[['concern_fake_sellers',
#     'concern_payment_fraud']].mean()


#STEP 3:
# WANTED FEATURES.
df['wanted_features_clean'] = (
    df['wanted_features']
    .str.lower()
    .str.replace('/', '')
    .str.strip()
)

# flag each feature for analysis.
df['feature_escrow'] = df['wanted_features_clean'].str.contains('escrow', na=False)
df['feature_verified_users'] = df['wanted_features_clean'].str.contains('verified', na=False)
df['feature_id_verification'] = df['wanted_features_clean'].str.contains('id verification', na=False)

#make them numerical for measurement.
df[['feature_escrow',
    'feature_verified_users',
    'feature_id_verification']] = \
df[['feature_escrow',
    'feature_verified_users',
    'feature_id_verification']].astype(int)

# step 4: willing to pay catagories and how.

df['price_willing_clean'] = (
    df['price_willing']
    .str.lower()
    .str.replace('–', '-')
    .str.strip()
)

#break each into a cartagory:

def categorize_price(x):
    if pd.isna(x):
        return 'unknown'
    
    x = x.lower()
    
    if 'wouldn’t pay' in x or 'r0' in x:
        return 'none'
    elif '%' in x:
        return 'percentage'
    elif 'r5' in x or 'r10' in x:
        return 'low'
    elif 'r20' in x:
        return 'high'
    else:
        return 'other'

df['price_category'] = df['price_willing_clean'].apply(categorize_price)

#go numerical.
def estimate_price(x):
    if pd.isna(x):
        return None
    
    x = x.lower()
    
    if 'wouldn’t pay' in x or 'r0' in x:
        return 0
    elif 'r5' in x and 'r10' in x:
        return 7
    elif 'r20' in x:
        return 20
    else:
        return None

df['price_estimate'] = df['price_willing_clean'].apply(estimate_price)




# step 5: when do you want to pay cleaning
df['when_pay_clean'] = (
    df['when_pay']
    .str.lower()
    .str.strip()
)

def categorize_payment(x):
    if pd.isna(x):
        return 'unknown'
    
    if 'before' in x:
        return 'before'
    elif 'after' in x:
        return 'after'
    else:
        return 'other'

df['payment_timing'] = df['when_pay_clean'].apply(categorize_payment)



# final cleaned file.

# Save clean dataset
df.to_csv("../data/cleaned/safetrade_cleaned.csv", index=False)
df.to_excel("../data/cleaned/safetrade_cleaned.xlsx", index=False)

/tmp/ipykernel_89263/1564401431.py:22: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:
